### 데이터 준비
1. CSV 파일에서 텍스트와 레이블을 읽음
    * TEXT_COL은 텍스트, LABEL_COL은 감정 레이블(0~5)을 의미함
2. 학습, 검증, 테스트 분할
    * train_test_split을 두 번 사용하여 학습(70%), 검증(15%), 테스트(15%)로 분리
    * stratify=labels 옵션으로 클래스 불균형에 대비해 레이블 분포를 보존함
    * random_state=SEED로 난수 고정, 재현성때문


### 텍스트 전처리 및 시퀀스 변환
1. 간단 토크나이저 (simple_tokenize)
    * 입력 텍스트를 소문자화(lower()), 줄바꿈을 공백으로 치환, 공백기준 분리하여 토큰 리스트로 반환
2. 단어집합(vocab) 생성 (build_vocab)
    * 학습 텍스트들에 대해 토큰 빈도 집계(Counter) 후 등장 빈도 상위 MAX_FEATURES 개 토큰을 선택
    * 사전 stoi 생성, 문자열(토큰)을 숫자 인덱스로 변환해주는 사전
    * itos는 숫자 인덱스를 문자열(토큰)로 역매핑
3. 텍스트를 시퀀스 변환 (text_to_sequence)
    * 각 텍스트를 토큰화 후 stoi로 인덱스 변환(사전에 없으면 <UNK>)
    * 시퀀스는 최대 길이 MAX_LEN로 자르고, 짧으면 <PAD>(인덱스 0)로 오른쪽 패딩
    * 결과는 길이 MAX_LEN인 정수 리스트


### 모델 학습용 데이터 인터페이스 통일
1. TextDataset 클래스 (PyTorch Dataset)
    * __len__ : 전체 샘플 수 반환
    * __getitem__(idx) : (sequence_tensor, label_tensor) 반환함
    *   시퀀스는 torch.long, 레이블도 torch.long(정수 레이블)
2. DataLoader
    * train_loader는 batch_size=BATCH_SIZE, shuffle=True로 배치 생성해서 학습 시 섞음
    * val_loader, test_loader는 평가용 배치를 순서대로 제공


### 포지셔널 인코딩 (PositionalEncoding)
1. 트랜스포머는 순서 정보를 내장하지 않으므로, 임베딩에 위치 정보를 더함
    * pe 텐서는 Positional Encoding을 모든 위치에 대해 미리 계산해둔 큰 표
    * 사인, 코사인 함수는 주기적이고 부드럽게 변해 위치 간의 상대적 거리 정보를 알 수 있음
    * 사인, 코사인의 주기적 특징 때문에 입력 길이가 달라도 위치 패턴이 유지됨
    * 고정된 수학적 패턴이 모델이 순서감각을 배우기 쉽게 해주어 학습 가능한 파라미터로 만들 필요가 없음
    * pe 텐서를 max_len x d_model로 만들어 사인/코사인 함수로 채움
    * pe[pos, 2i] = sin(pos / 10000^(2i/d_model))
    * pe[pos, 2i+1] = cos(pos / 10000^(2i/d_model))
    * pe에 unsqueeze(0) 해서 (1, max_len, d_model)로 만들고 register_buffer("pe", pe)로 모듈에 등록
    *   학습 가능한 파라미터가 아니고, 모델과 함께 저장/로딩되지만 업데이트되지 않음
2. forward(x)
    * 입력 x가 (batch_size, seq_len, d_model)일 때 x + pe[:, :seq_len, :]를 반환
    * 입력 x = 단어 의미를 나타내는 벡터, pe = 단어 위치를 나타내는 벡터라 둘을 더하면 1개의 벡터가 됨


### TransformerClassifier 모델 구조
1. Embedding 레이어
    * 입력 인덱스를 임베딩 벡터로 변환
    * 패딩 인덱스의 임베딩은 0으로 고정 처리해 optimizer가 갱신해도 변화 없음
    * 임베딩 가중치에 Xavier 초기화 적용
2. PositionalEncoding, 위에서 설명한 위치 인코딩을 더함
3. TransformerEncoder
    * 내부: TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ffn_dim, dropout=dropout, activation="relu")
    * 여러 레이어를 쌓음
4. Dropout + Linear로 최종 분류기 구현
5. 하이퍼파라미터
    * EMBED_DIM = 임베딩, 트랜스포머 차원
    * NHEAD = 멀티-헤드 어텐션 헤드 수
    * FFN_DIM = 트랜스포머 내부 피드포워드 숨은 차원
    * NUM_LAYERS = 인코더 레이어 반복 수
    * DROPOUT = 드롭아웃 비율
    * NUM_CLASSES = 출력 클래스 수


### Forward 연산 순서와 텐서 차원
0. 배치 크기 B, 시퀀스 길이 L, 임베딩 차원 D를 코드 순서대로 기술
1. 입력은 src
    * 형태는 (B, L) 정수 토큰 인덱스(패딩 포함)
2. 임베딩은 emb = self.embed(src)
    * 출력 형태는 (B, L, D)
3. 위치 인코딩 추가해서 emb = self.posenc(emb)
    * 형태는 (B, L, D)
4. Transformer 입력 형상 맞춤 emb_t = emb.transpose(0, 1)
    * nn.TransformerEncoder는 입력 형태가 (seq_len, batch_size, d_model)임
    * emb_t 형태는 (L, B, D)
5. 키 패딩 마스크 생성함 key_padding_mask = (src == 0)
    * 형태는 (B, L)로, 패딩 위치는 True가 됨(패딩 위치를 마스크)
6. Transformer 인코더 통과
    * out = self.transformer_encoder(emb_t, src_key_padding_mask=key_padding_mask)
    * 출력 형태는 (L, B, D)로 입력과 같은 형태
7. 다시 원래 차원으로 전치함 out = out.transpose(0, 1)
    * 형태는 (B, L, D)
8. 패딩을 무시한 평균 풀링(가중 평균) 수행함
    * mask = (~key_padding_mask).unsqueeze(-1).float()를 통해 패딩이 아닌 위치는 1.0, 패딩은 0.0. 형태는 (B, L, 1)
    * summed = (out * mask).sum(dim=1)를 통해 각 샘플의 비패딩 토큰들의 D차원 합 나타냄, 형태는 (B, D)
    * lengths = mask.sum(dim=1).clamp(min=1e-6)를 통해 각 샘플의 유효 길이(0 방지)를 나타냄, 형태는 (B, 1)
    * pooled = summed / lengths를 통해 각 샘플의 평균 임베딩(패딩 제외), 형태는 (B, D)
10.  토큰들의 평균을 취해 시퀀스 수준 표현으로 변환, mean pooling 사용함
11. 드롭아웃은 pooled = self.dropout(pooled)를 통해 수행함, 형태는 (B, D)
12. 최종 선형 분류기는 logits = self.fc(pooled)를 통해 수행, 출력 형태는 (B, NUM_CLASSES)
13. logits는 softmax 이전의 점수들


### Train 과정
1. 모델을 학습 모드로 전환함
2. 배치 반복
    * X: (B, L) 텐서, y: (B,) 레이블 정수 텐서
    * outputs = model(X)를 통해 (B, NUM_CLASSES) 출력
    * 손실은 loss = criterion(outputs, y), 코드에서는 CrossEntropyLoss(weight=weight_tensor) 사용
    * 클래스 불균형 보정을 위해 compute_class_weight로 얻은 가중치를 weight_tensor에 매핑하여 사용
    * 역전파 수행
    * 옵티마이저 업데이트
    * 배치 단위 총 손실 누적 후 epoch당 평균 손실 출력


### Evaluation 과정
1. 평가 모드 전환
2. torch.no_grad() 컨텍스트에서 전방 계산만 수행
    * 메모리, 속도 최적화
3. 각 배치에 대해 out = model(X)을 통해 pred = torch.argmax(out, dim=1) 계산
    * pred와 실제 y를 누적하여 전체 예측, 정답 리스트 구성
4. classification_report(trues, preds, digits=4)로 클래스별 precision, recall, f1-score, support 출력
5. confusion_matrix(trues, preds)로 오분류 패턴 확인


### 손실 함수와 클래스 불균형 보정
1. 학습 이전에 classes = np.unique(y_train)와 compute_class_weight(class_weight='balanced', classes=classes, y=y_train)로 클래스별 가중치 계산
2. weight_tensor를 NUM_CLASSES 길이로 만들고 각 클래스 인덱스 위치에 해당 가중치 할당
3. criterion = nn.CrossEntropyLoss(weight=weight_tensor)로 소손함수에 클래스 가중치를 넣어 소수 클래스의 손실 기여도를 증가시킴


### 전체 흐름
1. 랜덤 시드 고정 (random, numpy, torch)을 통해 재현성 확보
2. CSV에서 텍스트, 레이블 읽고 리스트로 변환
3. 학습, 검증, 테스트 분할 (train_test_split ×2, stratify 사용)
4. 학습 데이터로부터 vocabulary 생성 (build_vocab)해 stoi, itos 반환
5. TextDataset, DataLoader로 학습, 검증, 테스트 데이터 준비
6. TransformerClassifier 초기화(임베딩 초기화 포함)
7. 클래스 불균형에 대응하여 compute_class_weight로 가중치 계산하고 CrossEntropyLoss(weight=...) 준비
8. Adam 옵티마이저 초기화
9. 지정한 EPOCHS 만큼 반복
    * train_model: 배치 단위 학습(순전파 - 손실 - 역전파 - 가중치 갱신 순서대로)
    * evaluate로 검증셋 성능 출력(정밀도, 재현율, F1, 혼동행렬)
10. 학습 종료 후 테스트셋으로 최종 성능 출력


### 최종 성능
1. Train Loss은 0.4857에서 0.2945로 꾸준히 하락
    * 학습은 잘 진행됨
2. 검증 Accuracy는 Epoch1 0.8450에서 Epoch10 0.8670
    * 최종 테스트 accuracy 0.8699
    * 전반적으로 학습, 검증, 테스트에서 성능이 향상되었고 과적합 징후는 크지 않음
    * 검증 성능도 같이 향상됨
3. Macro F1은 Epoch1 ~0.8153에서 Epoch10 ~0.8311, Test ~0.8342
    * 클래스별 성능(특히 소수 클래스)을 고려한 향상도 있음
4. support 값
    * class0: 19048
    * class1: 22174
    * class2: 5430
    * class3: 9004
    * class4: 7513
    * class5: 2353
    * 클래스 불균형 존재함, class5가 상대적으로 작음
5. 최종 테스트 기준
    * class0: precision 0.9777, recall 0.8801, F1 0.9263으로 정확도 높고 안정적
    * class1: prec 0.9758, recall 0.8463, F1 0.9064로 정확하지만 일부 놓침
    * class2: prec 0.6911, recall 0.9326, F1 0.7939로 높은 재현율, 낮은 정밀도 (많이 맞추긴 하지만 예측이 과잉임)
    * class3: prec 0.7359, recall 0.8979, F1 0.8089로 재현율 높음, 정밀도 중간
    * class4: prec 0.8521, recall 0.8101, F1 0.8306로 균형 잡힌 편
    * class5: prec 0.6053, recall 0.9490, F1 0.7392로 매우 높은 재현율, 매우 낮은 정밀도
    * 거의 모든 진짜 class5는 잡지만, 예측한 것 중 많은 수가 틀림
6. 눈에 보이는 문제점
    * class1(실제)이 class2로 예측되는 케이스(1860개)와 class3로(876개)가 꽤 있음
    * class1과 class2/3 간 구별이 상대적으로 어려움
    * class4 실제가 class5로 오분류되는 케이스(766개)와 class3로(408개) 오분류되는 케이스 상당함
    * class4와 class5 사이, class4와 class3 사이 혼동 존재
    * class5 실제는 대부분 정확히 예측(2233/2353)되어 recall이 매우 높음, 하지만 반대로 예측된 class5의 많은 부분은 다른 클래스, 특히 class4에 속함(precision 낮음)
    * class2 실제는 비교적 잘 분류(5064/5430)됨, 찾는 능력은 좋음
7. Train Loss는 지속적으로 감소(0.4857에서 0.2945)해 모델이 안정적으로 학습하고 있음
8. Validation accuracy, F1도 증가해 과적합이 거의 없음
    * train loss 감소에도 val 성능 개선
9. 모델이 점점 더 민감해져(recall 증가) 많은 실제 양성 샘플을 잡지만, 동일하게 false positive을 줄이는 데는 한계가 있었음


### 특정 클래스 precision 낮은 이유
1.  클래스 간 표현 유사성 때문에 텍스트 패턴이 겹쳐 모델이 혼동할 수 있음
2. 레이블 노이즈, 라벨링 오류(특히 혼동되는 쌍) 존재 가능
3. 데이터 불균형 및 학습 목표
    * CrossEntropy + class-weight는 recall 향상에 도움되나 precision 개선은 보장 못 함
4. 토크나이저, 어휘 표현 한계
    * 가장 유력한 원인
    * 단순 whitespace 토크나이저 + 작은 vocab(MAX_FEATURES=1000)로 인해 구별 신호 손실
5. 풀링 방식의 한계
    * 평균풀링은 미묘한 어휘, 패턴 신호를 덜 집중해서 반영할 수 있음

In [ ]:
import random
import math
import numpy as np
import pandas as pd
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import torch.optim as optim

# configurations
DATA_PATH = "dataset/emotion_recognitions_merged.csv"
TEXT_COL = "text"
LABEL_COL = "label"
EPOCHS = 10
BATCH_SIZE = 64
LR = 1e-3
MAX_FEATURES = 1000
MAX_LEN = 100
EMBED_DIM = 128
NHEAD = 4
FFN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.1
NUM_CLASSES = 6
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# text preprocessing and vocabulary building
def simple_tokenize(text):
    text = str(text).lower()
    return text.replace('\n', ' ').split()

def build_vocab(texts, max_features=MAX_FEATURES, min_freq=1):
    counter = Counter()
    for t in texts:
        counter.update(simple_tokenize(t))

    most_common = [tok for tok, cnt in counter.most_common(max_features) if cnt >= min_freq]

    stoi = {"<PAD>": 0, "<UNK>": 1}
    for i, tok in enumerate(most_common, start=2):
        stoi[tok] = i

    itos = {i: s for s, i in stoi.items()}
    return stoi, itos

def text_to_sequence(text, stoi, max_len=MAX_LEN):
    tokens = simple_tokenize(text)
    seq = [stoi.get(t, stoi["<UNK>"]) for t in tokens][:max_len]
    seq += [stoi["<PAD>"]] * (max_len - len(seq))
    return seq

# Dataset
class TextDataset(Dataset):
    def __init__(self, texts, labels, stoi):
        self.texts = texts
        self.labels = labels
        self.stoi = stoi

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        seq = text_to_sequence(self.texts[idx], self.stoi)
        return torch.tensor(seq, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

# Transformer Classifier
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, num_heads=NHEAD,
                 ffn_dim=FFN_DIM, num_layers=NUM_LAYERS,
                 num_classes=NUM_CLASSES, dropout=DROPOUT, pad_idx=0):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.posenc = PositionalEncoding(embed_dim, max_len=MAX_LEN)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            activation="relu"
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(embed_dim, num_classes)

        nn.init.xavier_uniform_(self.embed.weight)

    def forward(self, src):
        emb = self.embed(src)
        emb = self.posenc(emb)

        emb_t = emb.transpose(0, 1)

        key_padding_mask = (src == 0)

        out = self.transformer_encoder(emb_t, src_key_padding_mask=key_padding_mask)
        out = out.transpose(0, 1)

        mask = (~key_padding_mask).unsqueeze(-1).float()
        summed = (out * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1e-6)
        pooled = summed / lengths

        pooled = self.dropout(pooled)
        logits = self.fc(pooled)
        return logits

# Train / Evaluate
def train_model(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0

    for X, y in loader:
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)

    return total_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    preds, trues = [], []

    with torch.no_grad():
        for X, y in loader:
            out = model(X)
            pred = torch.argmax(out, dim=1)
            preds.extend(pred.numpy())
            trues.extend(y.numpy())

    print(classification_report(trues, preds, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(trues, preds))

# Main
if __name__ == "__main__":
    df = pd.read_csv(DATA_PATH)
    texts = df[TEXT_COL].astype(str).tolist()
    labels = df[LABEL_COL].astype(int).tolist()

    X_train, X_tmp, y_train, y_tmp = train_test_split(
        texts, labels, test_size=0.3, stratify=labels, random_state=SEED
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED
    )

    stoi, itos = build_vocab(X_train, max_features=MAX_FEATURES)
    vocab_size = len(stoi)
    print("Vocab size:", vocab_size)

    train_ds = TextDataset(X_train, y_train, stoi)
    val_ds = TextDataset(X_val, y_val, stoi)
    test_ds = TextDataset(X_test, y_test, stoi)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

    model = TransformerClassifier(vocab_size=vocab_size)

    classes = np.unique(y_train)
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)

    weight_tensor = torch.zeros(NUM_CLASSES, dtype=torch.float)
    for i, c in enumerate(classes):
        weight_tensor[c] = class_weights[i]

    criterion = nn.CrossEntropyLoss(weight=weight_tensor)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        train_loss = train_model(model, train_loader, criterion, optimizer)
        print(f"\nEpoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}")
        print(f"Epoch {epoch+1} Validation:")
        evaluate(model, val_loader)

    print("\nFinal Test Performance:")
    evaluate(model, test_loader)

Vocab size: 1002


c:\Users\epoin\anaconda3\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(



Epoch 1/10 - Train Loss: 0.4857
Epoch 1 Validation:
              precision    recall  f1-score   support

           0     0.9708    0.8368    0.8988     19047
           1     0.9142    0.8390    0.8750     22174
           2     0.6979    0.9094    0.7897      5429
           3     0.8256    0.8508    0.8380      9004
           4     0.6731    0.8023    0.7321      7513
           5     0.6379    0.9333    0.7578      2354

    accuracy                         0.8450     65521
   macro avg     0.7866    0.8619    0.8153     65521
weighted avg     0.8630    0.8450    0.8492     65521

Confusion Matrix:
[[15938   765   157   862  1178   147]
 [  249 18605  1827   247   866   380]
 [   32   232  4937    73   134    21]
 [  112   452    84  7661   675    20]
 [   78   259    56   413  6028   679]
 [    8    39    13    23    74  2197]]

Epoch 2/10 - Train Loss: 0.3729
Epoch 2 Validation:
              precision    recall  f1-score   support

           0     0.9690    0.8603    0.9115